&nbsp;

## Antisense Transcript (UP-UP) Survey

- MALAT1 - TALAM1 の ような UP-UP case を調査する

In [1]:
import datetime
import os
import pathlib

import polars as pl
import polars.selectors as cs
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

## Prepare HN-score + fanta.bio

In [2]:
HNscore_all_fantabio = pl.read_parquet(
    "../../out/fantabio/parquet_file/HNscore_transcript_all_fantabio_v261001.parquet"
).filter(
    pl.col("cre_id").is_not_null() # fanta.bio の CRE が紐づかなかった場合は、今回は考慮しない
)

pl.Config.set_tbl_rows(2)
display(HNscore_all_fantabio)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000549545.1""",11,265,130,-254,8,174,69,-166,"""COQ10A""","""HGNC:26515""","""retained_intron""","""ENSG00000135469.15""",[],"""+""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_96252""","""chr12""",56268140,"""56268446""","""cp6@COQ10A""","[""ENST00000549545.1""]","""-23""",-23,"[{""EH38E4064794"",""pELS""}]",null,"[{""HS_00173371"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_02053667"",""Thyroid Gland"",""Thyroid gland"",""Others"",""gkmSVM""}]",null,null,-0.368421,"""ELA""","""255,202,74""","""yellow"""


&nbsp;

&nbsp;

## Prepare: ENCODE-rE2G Result (Heat Stress)

In [3]:
encode_rE2G_hs_result = pl.read_parquet(
    "../../out/ENCODE_rE2G/parquet_file/encode_re2g_hs_join_v261003.parquet"
).filter(
    pl.col("isSelfPromoter") == False
)


encode_rE2G_hs_result_unique_count = encode_rE2G_hs_result.group_by(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "target_cre_id",
    "target_cre_name",
    "target_fantabio_cre_class"
).agg(
    pl.col("re2g_dataset").unique().alias("re2g_dataset"),
    pl.col("re2g_dataset").n_unique().alias("re2g_dataset_count"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").alias("enhancer_cre_overlap_frac_of_fantabio_cre"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").min().alias("enhancer_cre_overlap_frac_of_fantabio_cre_min"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").max().alias("enhancer_cre_overlap_frac_of_fantabio_cre_max"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").mean().alias("enhancer_cre_overlap_frac_of_fantabio_cre_mean"), # 該当するデータセットで同定された enhancer と fantabio の CRE の重複率の平均値

    pl.col("ENCODE-rE2G.Score").alias("ENCODE-rE2G.Score"),
    pl.col("ENCODE-rE2G.Score").min().alias("ENCODE-rE2G.Score_min"),
    pl.col("ENCODE-rE2G.Score").max().alias("ENCODE-rE2G.Score_max"),
    pl.col("ENCODE-rE2G.Score").mean().alias("ENCODE-rE2G.Score_mean"), # 該当するデータセットで同定された enhancer と target promoter の相互作用のスコアの平均値

    # pl.col("ENCODE-rE2G.Score").quantile(0.25).alias("ENCODE-rE2G.Score_q1"),
    # pl.col("ENCODE-rE2G.Score").median().alias("ENCODE-rE2G.Score_median"),
    # pl.col("ENCODE-rE2G.Score").quantile(0.75).alias("ENCODE-rE2G.Score_q3"),
    # pl.col("ENCODE-rE2G.Score").std().alias("ENCODE-rE2G.Score_std"),
).sort(
    by=["re2g_dataset_count"],
    descending=True
)

print(f"ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level): {encode_rE2G_hs_result_unique_count.select(['cre_id', 'target_cre_id']).n_unique()}")
display(encode_rE2G_hs_result_unique_count)

ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level): 368398


cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64
"""FCHS_386999""","""FCHS_386999""","""PLA""","""FCHS_387005""","""cp1@VGF""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[1.0, 1.0, … 1.0]",0.741071,1.0,0.896811,"[0.937664, 0.935457, … 0.96808]",0.465491,0.96808,0.81646
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_130912""","""FCHS_130912""","""ELA""","""FCHS_489366""","""cp2@SLC25A47""","""ELA""","[""GSE280307_THP1_42C_120min""]",1,[1.0],1.0,1.0,1.0,[0.479222],0.479222,0.479222,0.479222


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Preparation: antisense transcript pair

- [antisense RNA process notebook](../06_transcript_characterization/06_5_antisense_RNA.ipynb) で作成したファイルを使用する
- GENCODE release 49 に紐づいている `tag` の選抜したリストでフィルタリング済み

In [4]:
bedfile_intersect_antisense_join = pl.read_parquet(
    "../../out/transcript_classification/antisense_transcript/antisense_transcript_join_HNscore_fantabio_v261010.parquet"
)

display(bedfile_intersect_antisense_join)

chr_a,start_a,end_a,transcript_id_a,strand_a,chr_b,start_b,end_b,transcript_id_b,strand_b,num_match,frac_match_a,frac_match_b,gene_id_a,gene_symbol_a,transcript_type_a,HN2_a,HN2_stranded_a,cre_info_a,n_cre_a,gene_id_b,gene_symbol_b,transcript_type_b,HN2_b,HN2_stranded_b,cre_info_b,n_cre_b,n_shared_cre,n_shared_cre_ELA,n_shared_cre_PLA
str,i64,i64,str,str,str,i64,i64,str,str,i64,f64,f64,str,str,str,i64,i64,list[struct[2]],u32,str,str,str,i64,i64,list[struct[2]],u32,u32,u32,u32
"""chr1""",765078,827475,"""ENST00000743835.1""","""-""","""chr1""",778746,810063,"""ENST00000412115.2""","""+""",31317,0.501899,1.0,"""ENSG00000225880.7""","""LINC00115""","""lncRNA""",-6,-1,"[{""FCHS_25"",""ELA""}, {""FCHS_26"",""PLA""}]",2,"""ENSG00000237491.11""","""LINC01409""","""lncRNA""",9,12,"[{""FCHS_19"",""PLA""}, {""FCHS_20"",""ELA""}]",2,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrX""",156023366,156025666,"""ENST00000496301.6""","""+""","""chrX""",156021995,156025666,"""ENST00000728197.1""","""-""",2300,1.0,0.626532,"""ENSG00000182484.15""","""WASH6P""","""retained_intron""",5,4,"[{""FCHS_446875"",""ELA""}]",1,"""ENSG00000295135.1""",null,"""lncRNA""",0,2,"[{""FCHS_593846"",""ELA""}]",1,0,0,0


In [5]:
# Canonical order so A-B and B-A collapse to one label.
pair_labels = (
    bedfile_intersect_antisense_join.select(
        pl.min_horizontal("transcript_id_a", "transcript_id_b").alias("id_lo"),
        pl.max_horizontal("transcript_id_a", "transcript_id_b").alias("id_hi"),
    )
    .unique(maintain_order=True)
    .select(pl.format("{} - {}", pl.col("id_lo"), pl.col("id_hi")).alias("pair"))
)

print(pair_labels.height)
pl.Config.set_tbl_rows(2)
display(pair_labels)

49887


pair
str
"""ENST00000412115.2 - ENST00000743835.1"""
…
"""ENST00000496301.6 - ENST00000728197.1"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Case 0: MALAT1 - TALAM1

In [6]:
# check (MALAT1 - TALAM1)
ENSG00000251562 = bedfile_intersect_antisense_join.filter(
    ((pl.col("gene_id_a").str.contains("ENSG00000251562")) & (pl.col("gene_id_b").str.contains("ENSG00000289740")))
    | ((pl.col("gene_id_a").str.contains("ENSG00000289740")) & (pl.col("gene_id_b").str.contains("ENSG00000251562")))
).sort(
    by=["HN2_stranded_a", "HN2_stranded_b"],
    descending=[True, True]
)

pl.Config.set_tbl_rows(4)
display(ENSG00000251562)

chr_a,start_a,end_a,transcript_id_a,strand_a,chr_b,start_b,end_b,transcript_id_b,strand_b,num_match,frac_match_a,frac_match_b,gene_id_a,gene_symbol_a,transcript_type_a,HN2_a,HN2_stranded_a,cre_info_a,n_cre_a,gene_id_b,gene_symbol_b,transcript_type_b,HN2_b,HN2_stranded_b,cre_info_b,n_cre_b,n_shared_cre,n_shared_cre_ELA,n_shared_cre_PLA
str,i64,i64,str,str,str,i64,i64,str,str,i64,f64,f64,str,str,str,i64,i64,list[struct[2]],u32,str,str,str,i64,i64,list[struct[2]],u32,u32,u32,u32
"""chr11""",65497605,65508073,"""ENST00000534336.4""","""+""","""chr11""",65499044,65507432,"""ENST00000698129.2""","""-""",8388,0.801299,1.0,"""ENSG00000251562.13""","""MALAT1""","""lncRNA""",74,56,"[{""FCHS_73693"",""ELA""}, {""FCHS_73694"",""PLA""}, {""FCHS_73695"",""PLA""}]",3,"""ENSG00000289740.2""","""TALAM1""","""lncRNA""",45,47,"[{""FCHS_73715"",""ELA""}, {""FCHS_470991"",""PLA""}, … {""FCHS_470992"",""ELA""}]",4,0,0,0
"""chr11""",65499044,65507432,"""ENST00000698129.2""","""-""","""chr11""",65497605,65508073,"""ENST00000534336.4""","""+""",8388,1.0,0.801299,"""ENSG00000289740.2""","""TALAM1""","""lncRNA""",45,47,"[{""FCHS_73715"",""ELA""}, {""FCHS_470991"",""PLA""}, … {""FCHS_470992"",""ELA""}]",4,"""ENSG00000251562.13""","""MALAT1""","""lncRNA""",74,56,"[{""FCHS_73693"",""ELA""}, {""FCHS_73694"",""PLA""}, {""FCHS_73695"",""PLA""}]",3,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr11""",65501786,65506516,"""ENST00000616691.2""","""+""","""chr11""",65499044,65507432,"""ENST00000698129.2""","""-""",4730,1.0,0.563901,"""ENSG00000251562.13""","""MALAT1""","""lncRNA""",-97,-49,"[{""FCHS_73702"",""PLA""}, {""FCHS_73703"",""PLA""}, {""FCHS_73704"",""PLA""}]",3,"""ENSG00000289740.2""","""TALAM1""","""lncRNA""",45,47,"[{""FCHS_73715"",""ELA""}, {""FCHS_470991"",""PLA""}, … {""FCHS_470992"",""ELA""}]",4,0,0,0
"""chr11""",65499050,65506468,"""ENST00000710862.1""","""+""","""chr11""",65499044,65507432,"""ENST00000698129.2""","""-""",7418,1.0,0.884359,"""ENSG00000251562.13""","""MALAT1""","""lncRNA""",-65,-64,"[{""FCHS_73696"",""PLA""}, {""FCHS_73697"",""PLA""}, {""FCHS_73698"",""PLA""}]",3,"""ENSG00000289740.2""","""TALAM1""","""lncRNA""",45,47,"[{""FCHS_73715"",""ELA""}, {""FCHS_470991"",""PLA""}, … {""FCHS_470992"",""ELA""}]",4,0,0,0


In [7]:
MALAT1_TALAM1_antisense = HNscore_all_fantabio.filter(
    (pl.col("transcript_id").is_in(ENSG00000251562.get_column("transcript_id_a").unique().implode()))
    | (pl.col("transcript_id").is_in(ENSG00000251562.get_column("transcript_id_b").unique().implode()))
).filter(
    pl.col("HN2_stranded") >= 40
)

pl.Config.set_tbl_rows(2)
display(MALAT1_TALAM1_antisense)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000534336.4""",105,31,270,74,70,14,167,56,"""MALAT1""","""HGNC:29665""","""lncRNA""","""ENSG00000251562.13""","[""TAGENE""]","""+""",1,"""URS0002A3A325""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","[""Assembled from PacBio CLS3 reads"", ""CLS3 project""]",2,"""FCHS_73693""","""chr11""",65496755,"""65497159""","""cp19@MALAT1""","[""ENST00000534336.4""]","""447""",447,"[{""EH38E2964425"",""pELS""}]","[{""ASCL1"",1208,[{""SRX8670125"",1208}, {""SRX8670130"",983}, … {""SRX8670129"",695}]}, {""TWIST1"",685,[{""SRX17998805"",685}]}, … {""TFAP2A"",575,[{""SRX1091550"",575}]}]","[{""HS_00156151"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, {""HS_00156152"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, … {""HS_00156154"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}]",null,"[{[""chr11:65496213-65497133""]}]",0.339286,"""ELA""","""255,202,74""","""yellow"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000698129.2""",95,50,261,45,60,13,178,47,"""TALAM1""","""HGNC:54476""","""lncRNA""","""ENSG00000289740.2""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""-""",4,"""URS0002A4D8F6""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","[""NR_145459.1 is the basis for this transcript. Built in response to a GENCODE ticket as RefSeq have this gene annotated. Discovered as RNA using coprecipitation and then characterised using strand specific PCR. There are two papers which deal with this gene but could not find any sequence with which to define the 5' and 3' ends.(PMID: 31382922) (PMID: 26826711). We got the raw RACE reads which I mapped and they looked OK"", ""CLS3 project""]",2,"""FCHS_470992""","""chr11""",65507911,"""65508440""","""cp9@TALAM1""","[""ENST00000698129.2"", ""NR_145459.1""]","""479""",479,"[{""EH38E2964449"",""pELS""}, {""EH38E2964450"",""dELS""}]","[{""ZNF865"",2204,[{""SRX13008929"",2204}, {""SRX13008930"",1262}]}, {""CEBPB"",2140,[{""SRX9029207"",2140}, {""SRX11379528"",1663}, … {""SRX11495275"",515}]}, … {""KDM5B"",504,[{""SRX265420"",504}]}]","[{""HS_00156151"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, {""HS_00156152"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, … {""HS_00156154"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}]",null,null,-0.117647,"""ELA""","""255,202,74""","""yellow"""


In [8]:
MALAT1_cre = MALAT1_TALAM1_antisense.filter(
    pl.col("gene_symbol") == "MALAT1"
).select("cre_id")

TALAM1_cre = MALAT1_TALAM1_antisense.filter(
    pl.col("gene_symbol") == "TALAM1"
).select("cre_id")

pl.Config.set_tbl_rows(2)
display(MALAT1_cre)
display(TALAM1_cre)

cre_id
str
"""FCHS_73693"""
…
"""FCHS_73695"""


cre_id
str
"""FCHS_73715"""
…
"""FCHS_470992"""


In [9]:
# test: MALAT1 - TALAM1 が、どの CRE の "target_cre_id" になっているかを確認
MALAT1_TALAM1_antisense = encode_rE2G_hs_result_unique_count.filter(
    (pl.col("target_cre_id").is_in(MALAT1_cre.get_column("cre_id").unique().implode()))
    | (pl.col("target_cre_id").is_in(TALAM1_cre.get_column("cre_id").unique().implode()))
).filter(
    pl.col("re2g_dataset_count") >= 14
)

pl.Config.set_tbl_rows(3)
display(MALAT1_TALAM1_antisense)

cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64
"""FCHS_73649""","""cp2@ENEMAL""","""PLA""","""FCHS_73694""","""cp7@MALAT1""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[1.0, 0.751124, … 0.931034]",0.751124,1.0,0.955879,"[0.605596, 0.560305, … 0.775636]",0.515692,0.935818,0.682022
"""FCHS_73669""","""cp2@LINC02736""","""PLA""","""FCHS_73694""","""cp7@MALAT1""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[0.611247, 0.611247, … 0.811736]",0.611247,1.0,0.696385,"[0.709414, 0.546621, … 0.736767]",0.374908,0.781765,0.594557
"""FCHS_73649""","""cp2@ENEMAL""","""PLA""","""FCHS_470991""","""cp8@TALAM1""","""PLA""","[""GSE101731_HFF_44C_120min"", ""GSE101731_HFF_44C_60min"", … ""GSE280307_THP1_42C_120min""]",14,"[1.0, 0.751124, … 0.931034]",0.751124,1.0,0.955879,"[0.324795, 0.287877, … 0.626757]",0.195157,0.626757,0.400493


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 4. UP-UP (stringent)

- 両方の transcript が HN-score (stranded) >= 100 である場合をフィルタリング
- 注目しやすい transcript pair を探すための簡単な条件

In [10]:
antisense_up_up = bedfile_intersect_antisense_join.filter(
    (pl.col("HN2_stranded_a") >= 100) & (pl.col("HN2_stranded_b") >= 100)
).sort(
    by=["HN2_stranded_a", "HN2_stranded_b"],
    descending=[True, True]
)

pl.Config.set_tbl_rows(32)
display(antisense_up_up)

chr_a,start_a,end_a,transcript_id_a,strand_a,chr_b,start_b,end_b,transcript_id_b,strand_b,num_match,frac_match_a,frac_match_b,gene_id_a,gene_symbol_a,transcript_type_a,HN2_a,HN2_stranded_a,cre_info_a,n_cre_a,gene_id_b,gene_symbol_b,transcript_type_b,HN2_b,HN2_stranded_b,cre_info_b,n_cre_b,n_shared_cre,n_shared_cre_ELA,n_shared_cre_PLA
str,i64,i64,str,str,str,i64,i64,str,str,i64,f64,f64,str,str,str,i64,i64,list[struct[2]],u32,str,str,str,i64,i64,list[struct[2]],u32,u32,u32,u32
"""chr17""",75135247,75154243,"""ENST00000476258.5""","""-""","""chr17""",75131088,75149037,"""ENST00000752388.1""","""+""",13790,0.725942,0.768288,"""ENSG00000189159.17""","""JPT1""","""protein_coding""",178,126,"[{""FCHS_506296"",""ELA""}, {""FCHS_182264"",""ELA""}, {""FCHS_182265"",""PLA""}]",3,"""ENSG00000298000.1""",null,"""lncRNA""",119,113,"[{""FCHS_182246"",""ELA""}, {""FCHS_182247"",""PLA""}, … {""FCHS_182249"",""PLA""}]",4,0,0,0
"""chr17""",75135285,75153293,"""ENST00000482348.5""","""-""","""chr17""",75131088,75149037,"""ENST00000752388.1""","""+""",13752,0.763661,0.766171,"""ENSG00000189159.17""","""JPT1""","""protein_coding""",197,116,"[{""FCHS_182259"",""ELA""}, {""FCHS_506295"",""ELA""}, … {""FCHS_182264"",""ELA""}]",5,"""ENSG00000298000.1""",null,"""lncRNA""",119,113,"[{""FCHS_182246"",""ELA""}, {""FCHS_182247"",""PLA""}, … {""FCHS_182249"",""PLA""}]",4,0,0,0
"""chr17""",75131088,75149037,"""ENST00000752388.1""","""+""","""chr17""",75135247,75154243,"""ENST00000476258.5""","""-""",13790,0.768288,0.725942,"""ENSG00000298000.1""",null,"""lncRNA""",119,113,"[{""FCHS_182246"",""ELA""}, {""FCHS_182247"",""PLA""}, … {""FCHS_182249"",""PLA""}]",4,"""ENSG00000189159.17""","""JPT1""","""protein_coding""",178,126,"[{""FCHS_506296"",""ELA""}, {""FCHS_182264"",""ELA""}, {""FCHS_182265"",""PLA""}]",3,0,0,0
"""chr17""",75131088,75149037,"""ENST00000752388.1""","""+""","""chr17""",75135285,75153293,"""ENST00000482348.5""","""-""",13752,0.766171,0.763661,"""ENSG00000298000.1""",null,"""lncRNA""",119,113,"[{""FCHS_182246"",""ELA""}, {""FCHS_182247"",""PLA""}, … {""FCHS_182249"",""PLA""}]",4,"""ENSG00000189159.17""","""JPT1""","""protein_coding""",197,116,"[{""FCHS_182259"",""ELA""}, {""FCHS_506295"",""ELA""}, … {""FCHS_182264"",""ELA""}]",5,0,0,0


In [11]:
# Canonical order so A-B and B-A collapse to one label.
pair_labels = (
    antisense_up_up.select(
        pl.min_horizontal("transcript_id_a", "transcript_id_b").alias("id_lo"),
        pl.max_horizontal("transcript_id_a", "transcript_id_b").alias("id_hi"),
    )
    .unique(maintain_order=True)
    .select(pl.format("{} - {}", pl.col("id_lo"), pl.col("id_hi")).alias("pair"))
)

print(pair_labels.height)
display(pair_labels)

2


pair
str
"""ENST00000476258.5 - ENST00000752388.1"""
"""ENST00000482348.5 - ENST00000752388.1"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Case 1: JPT1 - lncRNA transcript pair

- 両方の proximal CRE が Diff analysis (ATAC-seq); Diff-analysis (ChIP-seq; HSF1) の標的になっているか調査

In [17]:
up5p_diff_analysis = pl.read_parquet(
    "../../out/ChIP-Atlas_diff/diff_analysis_filter_parquet/transcript_cre/fantabio_HN2_up5p_v261002_diff_analysis_atac.parquet"
)

up5p_diff_analysis_jpt1_lncRNA = up5p_diff_analysis.filter(
    (pl.col("transcript_id").is_in(antisense_up_up.get_column("transcript_id_a").unique().implode()))
    | (pl.col("transcript_id").is_in(antisense_up_up.get_column("transcript_id_b").unique().implode()))
)

pl.Config.set_tbl_rows(3)
display(up5p_diff_analysis_jpt1_lncRNA)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000482348.5""",225,28,153,197,129,13,109,116,"""JPT1""","""HGNC:14569""","""protein_coding""","""ENSG00000189159.17""","[""alternative_5_UTR"", ""basic"", ""CCDS""]","""-""",3,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""alternative 5' UTR""]",1,"""FCHS_182264""","""chr17""",75153674,"""75154369""","""cp2@JPT1""","[""ENST00000476258.5"", ""ENST00000470924.5"", … ""BC039343""]","""0""",381,"[{""EH38E4171451"",""PLS""}, {""EH38E4171452"",""PLS""}]","[{""TP53BP1"",3000,[{""SRX3472375"",3000}]}, {""CTCF"",2722,[{""SRX10975999"",2722}, {""SRX10976001"",2322}, … {""SRX14609196"",551}]}, … {""TFEB"",502,[{""SRX6743819"",502}]}]","[{""HS_00234500"",""Small Intestine"",""Small intestine"",""Intestine"",""SVM""}, {""HS_00234501"",""Pancreas"",""Pancreas"",""Pancreas"",""SVM""}]","[{""hg19::chr17:73149941..73149971,-;hg_168043.1"",""p4@JPT1""}]",null,-0.376793,"""ELA""","""255,202,74""","""yellow""",75153673,75154369,[0.08046],"[{75154313,75154449,""chr17""}]",[1.971575],[5.5795e-11],[4.1085e-9],"[""GSE101731_atac_hff_44C_120min""]",1,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000476258.5""",205,27,174,178,130,4,117,126,"""JPT1""","""HGNC:14569""","""protein_coding""","""ENSG00000189159.17""","[""basic"", ""CCDS""]","""-""",2,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_182264""","""chr17""",75153674,"""75154369""","""cp2@JPT1""","[""ENST00000476258.5"", ""ENST00000470924.5"", … ""BC039343""]","""0""",0,"[{""EH38E4171451"",""PLS""}, {""EH38E4171452"",""PLS""}]","[{""TP53BP1"",3000,[{""SRX3472375"",3000}]}, {""CTCF"",2722,[{""SRX10975999"",2722}, {""SRX10976001"",2322}, … {""SRX14609196"",551}]}, … {""TFEB"",502,[{""SRX6743819"",502}]}]","[{""HS_00234500"",""Small Intestine"",""Small intestine"",""Intestine"",""SVM""}, {""HS_00234501"",""Pancreas"",""Pancreas"",""Pancreas"",""SVM""}]","[{""hg19::chr17:73149941..73149971,-;hg_168043.1"",""p4@JPT1""}]",null,-0.376793,"""ELA""","""255,202,74""","""yellow""",75153673,75154369,[0.08046],"[{75154313,75154449,""chr17""}]",[1.971575],[5.5795e-11],[4.1085e-9],"[""GSE101731_atac_hff_44C_120min""]",1,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000476258.5""",205,27,174,178,130,4,117,126,"""JPT1""","""HGNC:14569""","""protein_coding""","""ENSG00000189159.17""","[""basic"", ""CCDS""]","""-""",2,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_182265""","""chr17""",75154330,"""75154898""","""cp1@JPT1""","[""ENST00000409753.8"", ""ENST00000882069.1"", … ""AK312613""]","""0""",87,"[{""EH38E3243132"",""PLS""}, {""EH38E4171452"",""PLS""}]","[{""PBX1"",4441,[{""SRX9892487"",4441}, {""SRX9892489"",3774}, … {""SRX180167"",796}]}, {""PKNOX1"",3939,[{""SRX2424338"",3939}, {""SRX2424337"",3779}, … {""SRX2424711"",3254}]}, … {""RBL2"",500,[{""SRX11375741"",500}]}]","[{""HS_00234500"",""Small Intestine"

In [19]:
up5p_diff_analysis_hsf1 = pl.read_parquet(
    "../../out/ChIP-Atlas_diff/diff_analysis_filter_parquet/transcript_cre/fantabio_HN2_up5p_v261008_diff_analysis_chipseq_hsf1.parquet"
)

up5p_diff_analysis_hsf1_jpt1_lncRNA = up5p_diff_analysis_hsf1.filter(
    (pl.col("transcript_id").is_in(antisense_up_up.get_column("transcript_id_a").unique().implode()))
    | (pl.col("transcript_id").is_in(antisense_up_up.get_column("transcript_id_b").unique().implode()))
)

display(up5p_diff_analysis_hsf1_jpt1_lncRNA)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,motif_hit,n_motif_hit,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[struct[9]],u32,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Distal Enhancer Servey

- MALAT1 - TALAM1 pair に当てはめると、ENEMAL のように、どこか遠くの CRE から制御されている可能性がある

In [13]:
JPT1_antisense = HNscore_all_fantabio.filter(
    (pl.col("transcript_id").is_in(antisense_up_up.get_column("transcript_id_a").unique().implode()))
    | (pl.col("transcript_id").is_in(antisense_up_up.get_column("transcript_id_b").unique().implode()))
)

pl.Config.set_tbl_rows(2)
display(JPT1_antisense)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000482348.5""",225,28,153,197,129,13,109,116,"""JPT1""","""HGNC:14569""","""protein_coding""","""ENSG00000189159.17""","[""alternative_5_UTR"", ""basic"", ""CCDS""]","""-""",3,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""alternative 5' UTR""]",1,"""FCHS_182259""","""chr17""",75152686,"""75152838""","""cp4@JPT1""","[""ENST00000482348.5"", ""NM_001288610.1"", ""AK308253""]","""-455""",-455,"[{""EH38E3243128"",""pELS""}]",null,"[{""HS_00234500"",""Small Intestine"",""Small intestine"",""Intestine"",""SVM""}, {""HS_00234501"",""Pancreas"",""Pancreas"",""Pancreas"",""SVM""}]",null,null,-0.6,"""ELA""","""255,202,74""","""yellow"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000752388.1""",171,52,183,119,132,19,100,113,null,null,"""lncRNA""","""ENSG00000298000.1""","[""TAGENE""]","""+""",1,"""URS0002A5425C""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_182249""","""chr17""",75131554,"""75132056""","""cp2@NT5C""","[""ENST00000876511.1"", ""ENST00000245552.7"", … ""CU692564""]","""0""",-465,"[{""EH38E1885628"",""PLS""}, {""EH38E3243102"",""pELS""}]","[{""CTCF"",4411,[{""SRX10976008"",4411}, {""SRX8622035"",4158}, … {""SRX13007612"",505}]}, {""NRF1"",4193,[{""SRX2424732"",4193}, {""SRX2424029"",4182}, … {""SRX23881992"",516}]}, … {""TOX4"",509,[{""SRX9785101"",509}]}]",null,"[{""hg19::chr17:73127799..73127806,-;hg_168037.1"",""p6@NT5C""}, {""hg19::chr17:73127808..73127821,-;hg_168038.1"",""p2@NT5C""}, … {""hg19::chr17:73127900..73127911,-;hg_168040.1"",""p5@NT5C""}]",null,-0.971176,"""PLA""","""0,0,204""","""blue"""


In [14]:
JPT1_cre = JPT1_antisense.filter(
    pl.col("gene_id") == "ENSG00000189159.17"
).select("cre_id")

ENSG00000298000_cre = JPT1_antisense.filter(
    pl.col("gene_id") == "ENSG00000298000.1"
).select("cre_id")

pl.Config.set_tbl_rows(2)
display(JPT1_cre)
display(ENSG00000298000_cre)

cre_id
str
"""FCHS_182259"""
…
"""FCHS_182265"""


cre_id
str
"""FCHS_182246"""
…
"""FCHS_182249"""


In [15]:
# Case 0: どの CRE の "target_cre_id" になっているかを確認
JPT1_active_cre_target = encode_rE2G_hs_result_unique_count.filter(
    (pl.col("target_cre_id").is_in(JPT1_cre.get_column("cre_id").unique().implode()))
).sort(
    by=["cre_id"],
    descending=True
)

JPT_antisense_active_cre = encode_rE2G_hs_result_unique_count.filter(
    pl.col("target_cre_id").is_in(ENSG00000298000_cre.get_column("cre_id").unique().implode())
).sort(
    by=["cre_id"],
    descending=True
)

# 共通する CRE を確認
shared_cre_id = (
    JPT1_active_cre_target.select("cre_id").unique()
    .join(
        JPT_antisense_active_cre.select("cre_id").unique(),
        on="cre_id",
        how="inner",
    )
    .sort("cre_id")
)

JPT1_active_cre_target_filtered = JPT1_active_cre_target.filter(
    pl.col("cre_id").is_in(shared_cre_id.get_column("cre_id").unique().implode())
)

JPT_antisense_active_cre_filtered = JPT_antisense_active_cre.filter(
    pl.col("cre_id").is_in(shared_cre_id.get_column("cre_id").unique().implode())
)

pl.Config.set_tbl_rows(6)
print(shared_cre_id.height)
display(JPT1_active_cre_target_filtered)
display(JPT_antisense_active_cre_filtered)

6


cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64
"""FCHS_506274""","""FCHS_506274""","""ELA""","""FCHS_182265""","""cp1@JPT1""","""PLA""","[""GSE267931_HUVEC_39C_120min""]",1,[0.952381],0.952381,0.952381,0.952381,[0.208172],0.208172,0.208172,0.208172
"""FCHS_506273""","""FCHS_506273""","""ELA""","""FCHS_182265""","""cp1@JPT1""","""PLA""","[""GSE154744_K562_42C_30min"", ""GSE154744_K562_42C_30min_intact_hic"", ""GSE267931_HUVEC_39C_120min""]",3,"[0.97551, 0.97551, 1.0]",0.97551,1.0,0.983673,"[0.221308, 0.633988, 0.208172]",0.208172,0.633988,0.354489
"""FCHS_182187""","""FCHS_182187""","""PLA""","""FCHS_182265""","""cp1@JPT1""","""PLA""","[""GSE154744_K562_42C_12_5min"", ""GSE154744_K562_42C_30min"", … ""GSE267931_HUVEC_39C_120min""]",4,"[1.0, 1.0, … 1.0]",1.0,1.0,1.0,"[0.221402, 0.221308, … 0.208172]",0.208172,0.633988,0.321218
"""FCHS_182186""","""FCHS_182186""","""ELA""","""FCHS_182265""","""cp1@JPT1""","""PLA""","[""GSE154744_K562_42C_12_5min"", ""GSE154744_K562_42C_30min"", … ""GSE267931_HUVEC_39C_120min""]",4,"[1.0, 1.0, … 1.0]",1.0,1.0,1.0,"[0.221402, 0.221308, … 0.208172]",0.208172,0.633988,0.321218
"""FCHS_182185""","""FCHS_182185""","""PLA""","""FCHS_182265""","""cp1@JPT1""","""PLA""","[""GSE154744_K562_42C_12_5min"", ""GSE154744_K562_42C_30min"", … ""GSE267931_HUVEC_39C_120min""]",4,"[1.0, 1.0, … 1.0]",1.0,1.0,1.0,"[0.221402, 0.221308, … 0.208172]",0.208172,0.633988,0.321218
"""FCHS_182184""","""FCHS_182184""","""ELA""","""FCHS_182265""","""cp1@JPT1""","""PLA""","[""GSE154744_K562_42C_12_5min"", ""GSE154744_K562_42C_30min"", … ""GSE267931_HUVEC_39C_120min""]",4,"[1.0, 1.0, … 0.625821]",0.625821,1.0,0.906455,"[0.221402, 0.221308, … 0.208172]",0.208172,0.633988,0.321218


cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64
"""FCHS_506274""","""FCHS_506274""","""ELA""","""FCHS_182249""","""cp2@NT5C""","""PLA""","[""GSE267931_HUVEC_39C_120min""]",1,[0.952381],0.952381,0.952381,0.952381,[0.19448],0.19448,0.19448,0.19448
"""FCHS_506273""","""FCHS_506273""","""ELA""","""FCHS_182249""","""cp2@NT5C""","""PLA""","[""GSE267931_HUVEC_39C_120min""]",1,[1.0],1.0,1.0,1.0,[0.19448],0.19448,0.19448,0.19448
"""FCHS_182187""","""FCHS_182187""","""PLA""","""FCHS_182249""","""cp2@NT5C""","""PLA""","[""GSE267931_HUVEC_39C_120min""]",1,[1.0],1.0,1.0,1.0,[0.19448],0.19448,0.19448,0.19448
"""FCHS_182186""","""FCHS_182186""","""ELA""","""FCHS_182249""","""cp2@NT5C""","""PLA""","[""GSE267931_HUVEC_39C_120min""]",1,[1.0],1.0,1.0,1.0,[0.19448],0.19448,0.19448,0.19448
"""FCHS_182185""","""FCHS_182185""","""PLA""","""FCHS_182249""","""cp2@NT5C""","""PLA""","[""GSE267931_HUVEC_39C_120min""]",1,[1.0],1.0,1.0,1.0,[0.19448],0.19448,0.19448,0.19448
"""FCHS_182184""","""FCHS_182184""","""ELA""","""FCHS_182249""","""cp2@NT5C""","""PLA""","[""GSE267931_HUVEC_39C_120min""]",1,[0.625821],0.625821,0.625821,0.625821,[0.19448],0.19448,0.19448,0.19448
